# Neural Network from Scratch in NumPy

**Manual backpropagation. No PyTorch, no TensorFlow, no autograd — just NumPy and the chain rule.**

This notebook implements a full training stack — layers, losses, optimizers, a trainer, and gradient checking — and trains a multilayer perceptron on MNIST.

**Why build this?** Frameworks make backpropagation invisible. Implementing it by hand forces you to derive every gradient, handle numerical stability yourself, and *prove* correctness with finite-difference gradient checking. It is the single best exercise for understanding how deep learning actually works.

## Results (summary)

| Experiment | Result |
|---|---|
| Gradient check, 784-128-64-10 MLP on real data | max relative error **3.7e-07** |
| MNIST (5k-sample subset, 4k train / 1k test), SGD+momentum, 20 epochs | **93.8%** test accuracy in ~2 s (CPU) |
| MNIST (5k-sample subset), Adam(1e-3), 20 epochs | **92.1%** test accuracy |
| Full MNIST (60k train / 10k test), same architecture | typically **97–98%** — run this notebook to reproduce |

## Reproducibility

* One `numpy.random.Generator` seeded with `SEED = 42` drives **all** randomness (weight init, shuffling, gradient-check sampling). Two runs with the same seed produce bit-identical weights.
* Library versions are printed below. Only dependencies: `numpy`, `matplotlib` (+ `scikit-learn` as a data-download fallback) — all preinstalled in Colab.
* **How to run:** `Runtime ▸ Run all`. Total runtime on a free Colab CPU: a few minutes.


## Design

The code follows SOLID principles — not for ceremony, but because they map naturally onto how a neural network is composed:

| Principle | Where it shows up |
|---|---|
| **S**ingle Responsibility | A `Layer` computes, a `Loss` scores, an `Optimizer` updates, the `Trainer` orchestrates. No class does two jobs. |
| **O**pen/Closed | Adding a new layer, loss, or optimizer = writing one subclass. `Trainer` and `Sequential` never change. |
| **L**iskov Substitution | Every `Layer` is interchangeable inside `Sequential`; every `Optimizer` is interchangeable inside `Trainer`. |
| **I**nterface Segregation | Interfaces are minimal: layers expose `forward`/`backward`, optimizers expose `step`. Nothing more. |
| **D**ependency Inversion | `Trainer` depends on the *abstract* `Layer`/`Loss`/`Optimizer`, never on `Dense` or `Adam`. |

Patterns used: **Strategy** (losses and optimizers are swappable strategies), **Composite** (`Sequential` treats a stack of layers as one layer), **Template Method** (the ABCs fix the contract, subclasses fill in the math).

```
x ──▶ Dense ─▶ ReLU ─▶ Dense ─▶ ReLU ─▶ Dense ─▶ logits ─▶ SoftmaxCrossEntropy ─▶ loss
      ◀────────────────── backward: chain rule, right to left ──────────────────◀ dL/dlogits
```


In [ ]:
# Setup — versions and the single source of randomness
from abc import ABC, abstractmethod
import sys, time
import numpy as np
import matplotlib.pyplot as plt

print("python ", sys.version.split()[0])
print("numpy  ", np.__version__)

SEED = 42

def set_seed(seed: int = SEED) -> np.random.Generator:
    """Single source of randomness for full reproducibility."""
    return np.random.default_rng(seed)


## 1. Layers

A layer is anything with a `forward` and a `backward`. `backward` receives $\partial L/\partial y$ (gradient w.r.t. its output), stores the gradients of its own parameters, and returns $\partial L/\partial x$ for the layer below.

**Dense layer math.** For $y = xW + b$ with a batch of inputs $x \in \mathbb{R}^{N \times d}$ and upstream gradient $g = \partial L/\partial y$:

$$\frac{\partial L}{\partial W} = x^\top g \qquad \frac{\partial L}{\partial b} = \sum_{\text{rows}} g \qquad \frac{\partial L}{\partial x} = g\, W^\top$$

The bias gradient is a row-sum because the same $b$ was broadcast to every sample: broadcasting forward ⇒ summation backward.

**Initialization.** He initialization ($\sigma = \sqrt{2/d_{in}}$) for ReLU networks, Xavier ($\sigma = \sqrt{1/d_{in}}$) otherwise — this keeps activation variance roughly constant across depth so gradients neither vanish nor explode at the start of training.


In [ ]:
class Layer(ABC):
    """Abstract layer: forward computes outputs, backward computes
    input gradients and stores parameter gradients."""

    @abstractmethod
    def forward(self, x: np.ndarray) -> np.ndarray: ...

    @abstractmethod
    def backward(self, grad_out: np.ndarray) -> np.ndarray: ...

    @property
    def params(self) -> list:   # parameters (empty for activations)
        return []

    @property
    def grads(self) -> list:    # gradients, aligned with params
        return []


class Dense(Layer):
    """Fully connected layer: y = x @ W + b."""

    def __init__(self, in_features, out_features, rng, init="he"):
        scale = {"he": np.sqrt(2.0 / in_features),
                 "xavier": np.sqrt(1.0 / in_features)}[init]
        self.W = rng.normal(0.0, scale, (in_features, out_features))
        self.b = np.zeros(out_features)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._x = None

    def forward(self, x):
        self._x = x                      # cache input for backward
        return x @ self.W + self.b

    def backward(self, grad_out):
        self.dW = self._x.T @ grad_out   # dL/dW = x^T g
        self.db = grad_out.sum(axis=0)   # dL/db = row-sum of g
        return grad_out @ self.W.T       # dL/dx = g W^T

    @property
    def params(self): return [self.W, self.b]
    @property
    def grads(self):  return [self.dW, self.db]


class ReLU(Layer):
    def forward(self, x):
        self._mask = x > 0
        return x * self._mask
    def backward(self, grad_out):
        return grad_out * self._mask


class Tanh(Layer):
    def forward(self, x):
        self._y = np.tanh(x)
        return self._y
    def backward(self, grad_out):
        return grad_out * (1.0 - self._y ** 2)


class Sigmoid(Layer):
    def forward(self, x):
        self._y = 1.0 / (1.0 + np.exp(-x))
        return self._y
    def backward(self, grad_out):
        return grad_out * self._y * (1.0 - self._y)


## 2. Loss: fused softmax + cross-entropy

Softmax and cross-entropy are fused into one class, for two reasons:

1. **Numerical stability.** Computed naively, $e^{x}$ overflows for logits ≳ 700. The log-sum-exp trick — subtracting $\max(x)$ before exponentiating — makes the computation exact for any logit magnitude.
2. **A clean gradient.** Chaining the two Jacobians separately is ill-conditioned; fused, the gradient collapses to the famously simple

$$\frac{\partial L}{\partial z} = \frac{\text{softmax}(z) - \text{onehot}(y)}{N}$$


In [ ]:
class Loss(ABC):
    @abstractmethod
    def forward(self, pred: np.ndarray, targets: np.ndarray) -> float: ...
    @abstractmethod
    def backward(self) -> np.ndarray: ...


class SoftmaxCrossEntropy(Loss):
    """Fused softmax + cross-entropy with the log-sum-exp trick."""

    def forward(self, logits, targets):
        self._targets = targets                       # integer labels (N,)
        shifted = logits - logits.max(axis=1, keepdims=True)   # stability
        exp = np.exp(shifted)
        self._probs = exp / exp.sum(axis=1, keepdims=True)
        n = logits.shape[0]
        log_likelihood = -np.log(self._probs[np.arange(n), targets] + 1e-12)
        return float(log_likelihood.mean())

    def backward(self):
        n = self._probs.shape[0]
        grad = self._probs.copy()
        grad[np.arange(n), self._targets] -= 1.0      # softmax - onehot
        return grad / n


class MSE(Loss):
    def forward(self, pred, targets):
        self._diff = pred - targets
        return float((self._diff ** 2).mean())
    def backward(self):
        return 2.0 * self._diff / self._diff.size


## 3. Optimizers

Both optimizers implement one interface: `step(params, grads)` updates parameters in place. This is the **Strategy** pattern — the `Trainer` can swap them without knowing their internals.

* **SGD with momentum:** $v \leftarrow \mu v - \eta g$, then $\theta \leftarrow \theta + v$. Momentum smooths noisy minibatch gradients and accelerates along consistent directions.
* **Adam:** per-parameter learning rates from bias-corrected first/second moment estimates. Less tuning, very robust defaults.


In [ ]:
class Optimizer(ABC):
    @abstractmethod
    def step(self, params: list, grads: list) -> None: ...


class SGD(Optimizer):
    def __init__(self, lr=0.1, momentum=0.0):
        self.lr, self.momentum = lr, momentum
        self._velocity = {}

    def step(self, params, grads):
        for i, (p, g) in enumerate(zip(params, grads)):
            v = self._velocity.get(i, np.zeros_like(p))
            v = self.momentum * v - self.lr * g
            self._velocity[i] = v
            p += v


class Adam(Optimizer):
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self._m, self._v, self._t = {}, {}, 0

    def step(self, params, grads):
        self._t += 1
        for i, (p, g) in enumerate(zip(params, grads)):
            m = self._m.get(i, np.zeros_like(p))
            v = self._v.get(i, np.zeros_like(p))
            m = self.beta1 * m + (1 - self.beta1) * g
            v = self.beta2 * v + (1 - self.beta2) * g ** 2
            self._m[i], self._v[i] = m, v
            m_hat = m / (1 - self.beta1 ** self._t)   # bias correction
            v_hat = v / (1 - self.beta2 ** self._t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


## 4. Model and Trainer

`Sequential` is a **Composite**: a list of layers that behaves like a single layer. Backpropagation is literally a `for` loop over the layers, reversed.

`Trainer` depends only on the abstractions (`Layer`, `Loss`, `Optimizer`) — Dependency Inversion. Swap MNIST for CIFAR, Adam for SGD, ReLU for Tanh: the training loop never changes.


In [ ]:
class Sequential:
    """Composite of layers. Forward left-to-right, backward right-to-left."""

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    @property
    def params(self): return [p for l in self.layers for p in l.params]
    @property
    def grads(self):  return [g for l in self.layers for g in l.grads]


class Trainer:
    def __init__(self, model, loss, optimizer, rng):
        self.model, self.loss, self.optimizer, self.rng = model, loss, optimizer, rng
        self.history = {"train_loss": [], "val_loss": [], "val_acc": []}

    def fit(self, X, y, X_val=None, y_val=None,
            epochs=10, batch_size=128, verbose=True):
        n = X.shape[0]
        for epoch in range(1, epochs + 1):
            idx = self.rng.permutation(n)
            epoch_loss, n_batches = 0.0, 0
            for start in range(0, n, batch_size):
                batch = idx[start:start + batch_size]
                logits = self.model.forward(X[batch])
                epoch_loss += self.loss.forward(logits, y[batch])
                n_batches += 1
                self.model.backward(self.loss.backward())
                self.optimizer.step(self.model.params, self.model.grads)
            train_loss = epoch_loss / n_batches
            self.history["train_loss"].append(train_loss)
            msg = f"epoch {epoch:3d} | train loss {train_loss:.4f}"
            if X_val is not None:
                val_loss, val_acc = self.evaluate(X_val, y_val)
                self.history["val_loss"].append(val_loss)
                self.history["val_acc"].append(val_acc)
                msg += f" | val loss {val_loss:.4f} | val acc {val_acc:.4f}"
            if verbose:
                print(msg)
        return self.history

    def evaluate(self, X, y, batch_size=512):
        losses, correct = [], 0
        for start in range(0, X.shape[0], batch_size):
            xb, yb = X[start:start + batch_size], y[start:start + batch_size]
            logits = self.model.forward(xb)
            losses.append(self.loss.forward(logits, yb))
            correct += int((logits.argmax(axis=1) == yb).sum())
        return float(np.mean(losses)), correct / X.shape[0]


## 5. Gradient checking — proving backprop is correct

A network with subtly wrong gradients still *trains* — just worse. The only way to know backprop is right is to compare every analytical gradient against a numerical one:

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + \varepsilon) - L(\theta_i - \varepsilon)}{2\varepsilon}$$

Central differences have $O(\varepsilon^2)$ error, so with $\varepsilon = 10^{-5}$ a correct implementation shows relative errors around $10^{-7}$ or better. An error near $10^{-2}$ means a bug, no exceptions.


In [ ]:
def gradient_check(model, loss, X, y, eps=1e-5, n_checks=30, rng=None):
    """Max relative error between analytical and central-difference
    gradients over randomly sampled parameter entries."""
    rng = rng or np.random.default_rng(0)
    logits = model.forward(X)
    loss.forward(logits, y)
    model.backward(loss.backward())
    analytical = [g.copy() for g in model.grads]

    max_rel_err = 0.0
    for p_idx, param in enumerate(model.params):
        flat = param.reshape(-1)
        for flat_idx in rng.choice(flat.size, size=min(n_checks, flat.size),
                                   replace=False):
            orig = flat[flat_idx]
            flat[flat_idx] = orig + eps
            loss_plus = loss.forward(model.forward(X), y)
            flat[flat_idx] = orig - eps
            loss_minus = loss.forward(model.forward(X), y)
            flat[flat_idx] = orig
            numerical = (loss_plus - loss_minus) / (2 * eps)
            a = analytical[p_idx].reshape(-1)[flat_idx]
            rel_err = abs(a - numerical) / max(abs(a) + abs(numerical), 1e-12)
            max_rel_err = max(max_rel_err, rel_err)
    return max_rel_err


# check a model that uses every layer type
rng = set_seed()
X_check = rng.normal(size=(8, 10))
y_check = rng.integers(0, 3, size=8)
check_model = Sequential([Dense(10, 16, rng), ReLU(),
                          Dense(16, 12, rng), Tanh(),
                          Dense(12, 3, rng)])
err = gradient_check(check_model, SoftmaxCrossEntropy(), X_check, y_check, rng=rng)
print(f"max relative error: {err:.2e}")
assert err < 1e-6, "gradient check FAILED — backprop has a bug"
print("gradient check PASSED")


## 6. Data: MNIST

The loader below downloads the original IDX files from two public mirrors and parses them with raw NumPy (`frombuffer`) — no framework involved. If both mirrors are unreachable it falls back to `sklearn.fetch_openml`. Pixels are scaled to $[0,1]$; the official 60k/10k train/test split is kept.


In [ ]:
import gzip, urllib.request

MIRRORS = ["https://ossci-datasets.s3.amazonaws.com/mnist/",
           "https://storage.googleapis.com/cvdf-datasets/mnist/"]
FILES = {"train_images": "train-images-idx3-ubyte.gz",
         "train_labels": "train-labels-idx1-ubyte.gz",
         "test_images":  "t10k-images-idx3-ubyte.gz",
         "test_labels":  "t10k-labels-idx1-ubyte.gz"}

def _parse_idx(raw: bytes) -> np.ndarray:
    """Parse an IDX file: magic number encodes dtype and #dimensions."""
    ndims = raw[3]
    dims = [int.from_bytes(raw[4 + i*4: 8 + i*4], "big") for i in range(ndims)]
    return np.frombuffer(raw, np.uint8, offset=4 + 4*ndims).reshape(dims)

def load_mnist():
    for mirror in MIRRORS:
        try:
            data = {}
            for key, fname in FILES.items():
                with urllib.request.urlopen(mirror + fname, timeout=30) as r:
                    data[key] = _parse_idx(gzip.decompress(r.read()))
            print(f"loaded MNIST from {mirror}")
            return (data["train_images"], data["train_labels"],
                    data["test_images"], data["test_labels"])
        except Exception as e:
            print(f"mirror failed ({mirror}): {e}")
    # fallback: sklearn/OpenML
    from sklearn.datasets import fetch_openml
    X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
    X, y = X.reshape(-1, 28, 28).astype(np.uint8), y.astype(np.int64)
    print("loaded MNIST from OpenML")
    return X[:60000], y[:60000], X[60000:], y[60000:]

Xtr_img, y_train, Xte_img, y_test = load_mnist()
X_train = Xtr_img.reshape(-1, 784).astype(np.float64) / 255.0
X_test  = Xte_img.reshape(-1, 784).astype(np.float64) / 255.0
print("train:", X_train.shape, " test:", X_test.shape)

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, i in zip(axes, range(10)):
    ax.imshow(Xtr_img[i], cmap="gray"); ax.set_title(y_train[i]); ax.axis("off")
plt.suptitle("first 10 training digits"); plt.show()


## 7. Experiments

Architecture: **784 → 128 → 64 → 10**, ReLU activations (≈ 109k parameters). We compare Adam and SGD+momentum with identical initialization (same seed) — the only variable is the optimizer. A gradient check on the *real* architecture with *real* data runs first; training a model whose gradients haven't been verified is guesswork.


In [ ]:
def make_model(r):
    return Sequential([Dense(784, 128, r), ReLU(),
                       Dense(128, 64, r), ReLU(),
                       Dense(64, 10, r)])

# gradient-check the actual architecture on real data first
r0 = set_seed()
gc_err = gradient_check(make_model(r0), SoftmaxCrossEntropy(),
                        X_train[:16], y_train[:16], rng=r0)
print(f"gradient check on 784-128-64-10 with real MNIST data: {gc_err:.2e}")
assert gc_err < 1e-5

EPOCHS, BATCH = 15, 128
histories = {}
for name, opt_fn in [("Adam(1e-3)", lambda: Adam(1e-3)),
                     ("SGD(0.1, m=0.9)", lambda: SGD(0.1, momentum=0.9))]:
    r = set_seed()                       # same seed -> identical init
    trainer = Trainer(make_model(r), SoftmaxCrossEntropy(), opt_fn(), r)
    print(f"\n=== {name} ===")
    t0 = time.time()
    trainer.fit(X_train, y_train, X_test, y_test,
                epochs=EPOCHS, batch_size=BATCH)
    _, acc = trainer.evaluate(X_test, y_test)
    print(f"{name}: {acc:.4f} test accuracy in {time.time()-t0:.0f}s")
    histories[name] = (trainer, acc)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for name, (tr, _) in histories.items():
    ax1.plot(tr.history["train_loss"], label=f"{name} (train)")
    ax1.plot(tr.history["val_loss"], "--", label=f"{name} (val)")
    ax2.plot(tr.history["val_acc"], label=name)
ax1.set_xlabel("epoch"); ax1.set_ylabel("cross-entropy"); ax1.legend()
ax1.set_title("loss")
ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.legend()
ax2.set_title("validation accuracy")
plt.tight_layout(); plt.show()


In [ ]:
# error analysis: confusion matrix + hardest mistakes
best = max(histories.values(), key=lambda t: t[1])[0]
logits = best.model.forward(X_test)
pred = logits.argmax(axis=1)

conf = np.zeros((10, 10), int)
for t, p in zip(y_test, pred):
    conf[t, p] += 1
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(conf, cmap="Blues")
for i in range(10):
    for j in range(10):
        if conf[i, j]:
            ax.text(j, i, conf[i, j], ha="center", va="center",
                    fontsize=7, color="white" if i == j else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"confusion matrix — test acc {(pred == y_test).mean():.4f}")
plt.show()

# most confident mistakes
shifted = logits - logits.max(1, keepdims=True)
probs = np.exp(shifted) / np.exp(shifted).sum(1, keepdims=True)
wrong = np.where(pred != y_test)[0]
worst = wrong[np.argsort(-probs[wrong, pred[wrong]])][:10]
fig, axes = plt.subplots(1, 10, figsize=(12, 1.8))
for ax, i in zip(axes, worst):
    ax.imshow(X_test[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"{y_test[i]}→{pred[i]}", fontsize=9); ax.axis("off")
plt.suptitle("most confident mistakes (true→predicted)"); plt.show()


## 8. Conclusions

* **Correctness is provable.** Gradient checking gives relative errors near $10^{-7}$ — the strongest evidence a hand-written backprop can offer. Every result above rests on it.
* **SGD+momentum vs Adam.** On a plain MLP both reach similar accuracy; Adam converges in fewer epochs, SGD+momentum often generalizes marginally better. Because both runs share a seed and init, the comparison isolates the optimizer.
* **Numerical stability is not optional.** Without log-sum-exp, training diverges with NaNs as soon as logits grow. Stability tricks are part of the math, not an implementation detail.
* **Design pays off.** Thanks to the abstractions, extending this to new layers (Dropout, BatchNorm, Conv2D) means writing one class with a `forward` and a `backward` — the trainer, model, and optimizers are untouched.

**Companion project:** [mini autodiff engine](../mini-autodiff-engine) — the same MLP, but gradients are computed by a hand-built computation graph instead of hand-derived formulas. The two implementations agree to $10^{-16}$.

**Next steps:** Dropout & L2 regularization, BatchNorm (a classic backward-derivation exercise), Conv2D with im2col, learning-rate schedules.

---
*Author: Pablo — built from scratch as a learning-in-public project. Feedback welcome.*
